# 04 – Baseline Models
Majority class, keyword baseline, resume-only leakage check, Logistic Regression and Random Forest. Trained on train, compared on validation; test is not used here.

## Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd, numpy as np, os, time, joblib
from datetime import datetime
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report

base = "/content/drive/MyDrive/resume-jd-matching"
data_dir, model_dir = f"{base}/data", f"{base}/models"
os.makedirs(f"{base}/reports", exist_ok=True)

train = pd.read_parquet(f"{data_dir}/train_feat.parquet")
val = pd.read_parquet(f"{data_dir}/val_feat.parquet")
y_tr, y_va = train["label"], val["label"]
order = ["No Fit", "Potential Fit", "Good Fit"]

FEATS = ["skill_coverage", "skill_jaccard", "matched_count", "missing_count", "jd_skill_count",
         "resume_skill_count", "jd_years", "resume_years", "exp_gap", "exp_meets",
         "jd_years_missing", "resume_years_missing", "tfidf_cosine", "embed_cosine"]
print(train.shape, val.shape, "| features:", len(FEATS))

Mounted at /content/drive
(5285, 19) (943, 19) | features: 14


## Evaluation and experiment log

In [3]:
log_path = f"{base}/reports/experiments.csv"

def evaluate(name, model, Xtr, Xva, features, params="", verbose=True):
    t = time.time(); model.fit(Xtr, y_tr); fit_s = time.time() - t
    pred = model.predict(Xva)
    f1, acc = f1_score(y_va, pred, average="macro"), accuracy_score(y_va, pred)
    row = {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "notebook": "04", "model": name,
           "features": features, "params": params, "val_macro_f1": round(f1, 4),
           "val_accuracy": round(acc, 4), "fit_seconds": round(fit_s, 1)}
    pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
    print(f"{name}: macro F1 {f1:.3f} | accuracy {acc:.3f}")
    if verbose:
        print(classification_report(y_va, pred, labels=order, digits=3, zero_division=0))
    return f1, model

## Baseline 1: majority class

In [4]:
evaluate("Majority class", DummyClassifier(strategy="most_frequent"),
         train[FEATS].fillna(0), val[FEATS].fillna(0), "none");

Majority class: macro F1 0.218 | accuracy 0.486
               precision    recall  f1-score   support

       No Fit      0.486     1.000     0.654       458
Potential Fit      0.000     0.000     0.000       233
     Good Fit      0.000     0.000     0.000       252

     accuracy                          0.486       943
    macro avg      0.162     0.333     0.218       943
 weighted avg      0.236     0.486     0.318       943



## Baseline 2: keyword (TF-IDF cosine only)

In [5]:
evaluate("Keyword baseline", make_pipeline(StandardScaler(),
         LogisticRegression(class_weight="balanced", max_iter=1000)),
         train[["tfidf_cosine"]], val[["tfidf_cosine"]], "tfidf_cosine");

Keyword baseline: macro F1 0.408 | accuracy 0.475
               precision    recall  f1-score   support

       No Fit      0.615     0.531     0.570       458
Potential Fit      0.308     0.103     0.154       233
     Good Fit      0.385     0.718     0.501       252

     accuracy                          0.475       943
    macro avg      0.436     0.451     0.408       943
 weighted avg      0.478     0.475     0.449       943



## Leakage check: resume text only (no JD)

In [6]:
evaluate("Resume-only check", make_pipeline(
         TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True, max_features=20000),
         LogisticRegression(class_weight="balanced", max_iter=2000)),
         train["resume"], val["resume"], "resume text only");

Resume-only check: macro F1 0.426 | accuracy 0.442
               precision    recall  f1-score   support

       No Fit      0.650     0.469     0.545       458
Potential Fit      0.303     0.378     0.337       233
     Good Fit      0.354     0.452     0.397       252

     accuracy                          0.442       943
    macro avg      0.436     0.433     0.426       943
 weighted avg      0.485     0.442     0.454       943



## Logistic Regression (all features, C tuned on validation)

In [7]:
lr_runs = []
for C in [0.01, 0.1, 1, 10]:
    m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(C=C, class_weight="balanced", max_iter=2000))
    lr_runs.append(evaluate(f"LogReg C={C}", m, train[FEATS], val[FEATS], "all 14", f"C={C}", verbose=False) + (C,))
best_lr_f1, best_lr, best_C = max(lr_runs, key=lambda r: r[0])
print(f"\nBest LogReg: C={best_C}, macro F1 {best_lr_f1:.3f}")
print(classification_report(y_va, best_lr.predict(val[FEATS]), labels=order, digits=3))

LogReg C=0.01: macro F1 0.463 | accuracy 0.476
LogReg C=0.1: macro F1 0.460 | accuracy 0.473
LogReg C=1: macro F1 0.459 | accuracy 0.472
LogReg C=10: macro F1 0.459 | accuracy 0.472

Best LogReg: C=0.01, macro F1 0.463
               precision    recall  f1-score   support

       No Fit      0.679     0.489     0.569       458
Potential Fit      0.340     0.498     0.404       233
     Good Fit      0.401     0.433     0.416       252

     accuracy                          0.476       943
    macro avg      0.473     0.473     0.463       943
 weighted avg      0.521     0.476     0.487       943



## Random Forest (all features, small grid on validation)

In [8]:
rf_runs = []
for depth in [None, 10]:
    for leaf in [1, 5, 20]:
        m = make_pipeline(SimpleImputer(strategy="median"),
                          RandomForestClassifier(n_estimators=300, max_depth=depth, min_samples_leaf=leaf,
                                                 class_weight="balanced", n_jobs=-1, random_state=42))
        p = f"depth={depth}, leaf={leaf}"
        rf_runs.append(evaluate(f"RandomForest {p}", m, train[FEATS], val[FEATS], "all 14", p, verbose=False) + (p,))
best_rf_f1, best_rf, best_p = max(rf_runs, key=lambda r: r[0])
print(f"\nBest RandomForest: {best_p}, macro F1 {best_rf_f1:.3f}")
print(classification_report(y_va, best_rf.predict(val[FEATS]), labels=order, digits=3))

imp = pd.Series(best_rf[-1].feature_importances_, index=FEATS).sort_values(ascending=False)
print("\nFeature importance (Random Forest):\n", imp.round(3).to_string())

RandomForest depth=None, leaf=1: macro F1 0.485 | accuracy 0.549
RandomForest depth=None, leaf=5: macro F1 0.498 | accuracy 0.520
RandomForest depth=None, leaf=20: macro F1 0.478 | accuracy 0.491
RandomForest depth=10, leaf=1: macro F1 0.490 | accuracy 0.511
RandomForest depth=10, leaf=5: macro F1 0.486 | accuracy 0.501
RandomForest depth=10, leaf=20: macro F1 0.483 | accuracy 0.495

Best RandomForest: depth=None, leaf=5, macro F1 0.498
               precision    recall  f1-score   support

       No Fit      0.721     0.566     0.634       458
Potential Fit      0.347     0.605     0.441       233
     Good Fit      0.506     0.357     0.419       252

     accuracy                          0.520       943
    macro avg      0.525     0.509     0.498       943
 weighted avg      0.571     0.520     0.529       943


Feature importance (Random Forest):
 tfidf_cosine            0.131
embed_cosine            0.128
jd_skill_count          0.104
jd_years                0.088
skill_jaccard

## Save best models and experiment summary

In [9]:
joblib.dump(best_lr, f"{model_dir}/logreg.joblib")
joblib.dump(best_rf, f"{model_dir}/random_forest.joblib")
log = pd.read_csv(log_path)
print(log[log["notebook"] == 4].sort_values("val_macro_f1", ascending=False)
      [["model", "params", "val_macro_f1", "val_accuracy"]].to_string(index=False))

                           model              params  val_macro_f1  val_accuracy
 RandomForest depth=None, leaf=5  depth=None, leaf=5        0.4980        0.5196
   RandomForest depth=10, leaf=1    depth=10, leaf=1        0.4898        0.5111
   RandomForest depth=10, leaf=5    depth=10, leaf=5        0.4863        0.5005
 RandomForest depth=None, leaf=1  depth=None, leaf=1        0.4848        0.5493
  RandomForest depth=10, leaf=20   depth=10, leaf=20        0.4831        0.4952
RandomForest depth=None, leaf=20 depth=None, leaf=20        0.4776        0.4910
                   LogReg C=0.01              C=0.01        0.4629        0.4761
                    LogReg C=0.1               C=0.1        0.4596        0.4730
                      LogReg C=1                 C=1        0.4587        0.4719
                     LogReg C=10                C=10        0.4587        0.4719
               Resume-only check                 NaN        0.4262        0.4422
                Keyword base

## Baseline findings
- Validation macro F1: Majority 0.218, Keyword 0.408, Resume-only 0.426, LogReg 0.463, Random Forest 0.498.
- Resume-only check (no JD) reaches 0.426: labels partly depend on which resume it is (resumes are heavily reused). This is a dataset property; the real bar for matching ability is 0.426.
- Random Forest beats the resume-only check by 0.07 using only JD–resume comparison features → genuine matching signal.
- RF > LogReg → non-linear interactions help.
- Hardest distinction: Potential Fit vs Good Fit (Good Fit recall 0.36).